# IMDB sentiment classification

**notebook by <span style="color:#75ACF0">b0nba</span>**

This notebook is a simple piece of work and it is my first dive into NLP and text classification.

For this work I have set myself **30 days deadline** and during that time I was trying to code and thoroughly understand tools used. For this reason presented code is basic and minimal. <br>
<span style="color:red">**It lacks**</span> proper EDA, deeper data cleaning, multiple models testing, evaluation and fine-tuning.
I have also left couple of unused cells in order to <span style="color:green">**comeback**</span> to this code again in the future.

For the most part I would treat this notebook as beginners attempt for NLP and text classification.

Any <span style="color:red">critique</span> and <span style="color:green">suggestions</span> are welcomed :)

Let us import all modules that will be used.

In [1]:
import kagglehub
import pandas as pd
import regex as re
from bs4 import BeautifulSoup
import spacy as sp

/home/b0nba/Praca/Programming/Programming/Projects/NLP/IMDB/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Install english model - for remote sessions.

In [ ]:
!python -m spacy download en_core_web_md

Next we load the data.

# Load the Data

In [2]:
# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
path = path + "/IMDB Dataset.csv"

print("Path to dataset files:", path)

Path to dataset files: /home/b0nba/.cache/kagglehub/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/versions/1/IMDB Dataset.csv


Assing our data to a dataframe and our 'path' serves as backup.

In [3]:
df = pd.read_csv(path)

We start as standard with data cleaning. This is very important step which normally would require carefull examination of the data - in order to assess that should be removed/changed and what not. In this notebook I used pretty standard framework for data cleaning and normalization, it is just worth to note that it could be examined little more in the fine tuning process which this notebook lacks.

# Data cleaning

##### Check for duplicates

In [4]:
df = df.drop_duplicates()
df['sentiment'].value_counts()

sentiment
positive    24884
negative    24698
Name: count, dtype: int64

Duplicates were removed, as they can lead to overfitting the model.

##### Check for a missing values

In [5]:
df[df.isna().any(axis=1)]

,review,sentiment


##### Text normalization

###### Set letters to lowercase

In [6]:
df['review'] = df['review'].apply(lambda x: x.lower() )

###### Remove whitespaces

In [7]:
print(df['review'][15])
#df['review'] = df['review'].apply(lambda x: x.strip())

kind of drawn in by the erotic scenes, only to realize this was one of the most amateurish and unbelievable bits of film i've ever seen. sort of like a high school film project. what was rosanna arquette thinking?? and what was with all those stock characters in that bizarre supposed midwest town? pretty hard to get involved with this one. no lessons to be learned from it, no brilliant insights, just stilted and quite ridiculous (but lots of skin, if that intrigues you) videotaped nonsense....what was with the bisexual relationship, out of nowhere, after all the heterosexual encounters. and what was with that absurd dance, with everybody playing their stereotyped roles? give this one a pass, it's like a million other miles of bad, wasted film, money that could have been spent on starving children or aids in africa.....


###### Remove HTML, urls and mails

In [8]:
df['review'][1]

'a wonderful little production. <br /><br />the filming technique is very unassuming- very old-time-bbc fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />the actors are extremely well chosen- michael sheen not only "has got all the polari" but he has all the voices down pat too! you can truly see the seamless editing guided by the references to williams\' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. a masterful production about one of the great master\'s of comedy and his life. <br /><br />the realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional \'dream\' techniques remains solid then disappears. it plays on our knowledge and our senses, particularly with the scenes concerning orton and halliwell and the sets (particularly of their flat with halliwell\'s murals decorating every surface) are terribly well d

In [9]:
# HTML
df['review'] = df['review'].apply(lambda x: BeautifulSoup(x, "html.parser").text)
df['review'][1]

'a wonderful little production. the filming technique is very unassuming- very old-time-bbc fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. the actors are extremely well chosen- michael sheen not only "has got all the polari" but he has all the voices down pat too! you can truly see the seamless editing guided by the references to williams\' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. a masterful production about one of the great master\'s of comedy and his life. the realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional \'dream\' techniques remains solid then disappears. it plays on our knowledge and our senses, particularly with the scenes concerning orton and halliwell and the sets (particularly of their flat with halliwell\'s murals decorating every surface) are terribly well done.'

###### Remove special characters

In [10]:
df['review'] = df['review'].apply(lambda x: re.sub('[^a-zA-Z]', ' ', x))
df['review'][2]


'i thought this was a wonderful way to spend time on a too hot summer weekend  sitting in the air conditioned theater and watching a light hearted comedy  the plot is simplistic  but the dialogue is witty and the characters are likable  even the well bread suspected serial killer   while some may be disappointed when they realize this is not match point    risk addiction  i thought it was proof that woody allen is still fully in control of the style many of us have grown to love this was the most i d laughed at one of woody s comedies in years  dare i say a decade    while i ve never been impressed with scarlet johanson  in this she managed to tone down her  sexy  image and jumped right into a average  but spirited young woman this may not be the crown jewel of his career  but it was wittier than  devil wears prada  and more interesting than  superman  a great comedy to go see with friends '

##### Tokenize

In [11]:
nlp = sp.load("en_core_web_md")

texts = df["review"].tolist()
docs = nlp.pipe(texts, batch_size=50, n_process=4)

In [12]:
docs = list(docs)

##### Remove stopwords

In [13]:
results = []
for doc in docs:
    result = [token.text for token in doc if not token.is_stop]
    results.append(result)

# Feature Engineering

## Surface features

Initially I thought that **adding surface features** into a model would increase accuracy. My hypothesis was that maybe people who enjoyed the movie will write more about it. This is not included in this notebook but this was tested, and **it did not contribute** for improvement of model accuracy at all.

I leave it as is so I can maybe do something more useful with it in the future

### Sentence count

In [14]:
df['n_sentences'] = [len(list(doc.sents)) for doc in docs]

### Word count

In [15]:
word_amount = [len(list(doc)) for doc in docs]
df['word_count'] = word_amount / df['n_sentences']

### Character count

In [16]:
df['char_count'] = df["review"].str.len()
df['char_count'] = df['char_count'] / word_amount

## Other features

Due to the deadline that I have set for myself I did not managed to incorporate it. I leave it as is because I believe this could be potentially improved maybe in the future.

### Consider:
1. Actor name. Better rated actors can influence positive movie score
2. Director name. Better rated directors can influence positive movie score even more

# Vectorization

Nothing to crazy going on here. I have decided to use TF-IDF embedder as I wanted to keep this notebook as plain and simple as possible - as this is my first public work.

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer


vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=0.9,
    min_df=2,
    ngram_range=(1, 2)
)

texts_clean = [" ".join(tokens) for tokens in results]

texts_train, texts_test, y_train, y_test = train_test_split(
    texts_clean, df["sentiment"], test_size=0.1, random_state=42
)

X_train = vectorizer.fit_transform(texts_train)
X_test = vectorizer.transform(texts_test)

# Modeling

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    negative       0.91      0.87      0.89      2487
    positive       0.88      0.91      0.89      2472

    accuracy                           0.89      4959
   macro avg       0.89      0.89      0.89      4959
weighted avg       0.89      0.89      0.89      4959



This notebook obviously lacks model evaluation and fine-tuning. I suspect this will be performed in the future works, after I get better grasp on modules and available tools - I would be able to work faster.

**<p style="font-size:24px"> The End :)</p>**

Thank you for your time ;)